

### 1. Error Detection Categories: 

---

**TOX (Toxicity):**  
Deviation in toxicity (hate, violence, or profanity) against an individual or group (e.g., religion, race, gender).  
*Why critical?* Toxic content can cause harm, incite violence or cause hate between different groups of people, which could lead to serious conflicts.

*Example 1:*  
- Source: "The president is a greatly admired leader."
- Back-translated MT could be: "The president is a greatly dreaded leader." 
- Reason: (Causing hate or affecting certain groups or more if translated incorrectly)

*Example 2:*  
- source: “Stop wasting valuable Internet resources (you’re hurting the planet)”
- Back-translated MT: “Stop wasting valuable Internet resources (may the  planet kill you)”
- Reason: critical mistranslation into violent content.

---

**SAF (Health or Safety Risks):**  
Deviation in health or safety risks, where translation errors may endanger the reader.  
*Why critical?* Incorrect health or safety information can lead to physical harm or dangerous actions.

*Example 1:*  
- Source: "Take two pills daily."  
- Back-translated MT: "Take twenty pills daily." (overdose affecting the patient life)

*Example 2:*  
- Source: “Wash your hands, or you will catch the coronavirus”
- Back-translated MT: “Shake hands, or you will catch the coronavirus”
- Reason: critical mistranslation, adding a safety risk.


---

**NAM (Named Entities):**  
Deviation in named entities (people, organizations, locations, etc.), such as deletion, mistranslation by another NE or common word. Or translated where the transliteration makes no sense in the target language.
*Why critical?* Misidentifying entities can cause confusion, legal issues, or misdirected actions.

*Example 1:*  
- Source: "Professor Smith, your supervisor, lives in Berlin. Go there and call him to pick you up"  
- Back-translated MT: "Professor Smith, your supervisor, lives in Stockholm. Go there and call him to pick you up" (Incorrect NE used, which could cause great confusion, waste time/money/effort.)

*Example 2:*  

- Source: “They are the worst band!”
- Back-translated MT: “Cold Play are the worst band!”
- Reason: critical introduction of named entity.
---

**SEN (Sentiment Polarity or Negation):**  
Deviation in sentiment or negation, where the translation reverses or alters the intended sentiment (removes/add/ negation or alter confidence rate).  
*Why critical?* Misrepresenting sentiment can damage reputations, relationships, or lives.  

*Example 1:*  
- Source: "The results are not promising."
- Back-translated MT: "The results are promising." (Negation removed, introducing false/misleading facts, e.g. percentage of a surgery success rate low becoming high or vice-versa) 

*Example 2:*  

- Source: “I never wrote this article, I just edited it”  
- Back-translated MT: “I never wrote this article, I never edited it”  
- Reason: critical mistranslation, the second clause was negated in the translation.
---

**NUM (Units/Time/Date/Numbers):**  
Deviation in numbers, units, dates, or times, such as incorrect translation or omission.  
*Why critical?* Errors can lead to missed appointments, financial loss, or logistical failures.  

*Example 1:*  
- Source: "The meeting is on July 10 at 3 PM."  
- Back-translated MT: "The meeting is on July 11 at 8 PM." (Incorrect date and time, which could cause great confusion, waste time/money/effort.)

*Example 2:*  
- Source: “From that point, turn right and drive 20 kilometers”
- Back-translated MT: “From that point, turn right and drive 20 miles”
- Reason: critical mistranslation as it leads to incorrect directions.

### Installing required Env:

In [3]:
pip install -U scikit-learn transformers pandas 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118Note: you may need to restart the kernel to use updated packages.




[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install --upgrade transformers[torch] accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch 

# Sanity Check for device usage
torch.cuda.is_available()
torch.cuda.get_device_name()

'NVIDIA GeForce GTX 1060 with Max-Q Design'

### Preprocessing and data prep 

In [ ]:
import pandas as pd
from sklearn.utils import shuffle

# Load and concatenate train/dev files for all 4 languages
lang_prefixes = ["encs", "ende", "enja", "enzh"]
# lang_prefixes = ["enzh"]
train_dfs = []
dev_dfs = []

for prefix in lang_prefixes:
    train_file = f"{prefix}_majority_train.tsv"
    dev_file = f"{prefix}_majority_dev.tsv"
    train_df_lang = pd.read_csv(train_file, sep="\t", header=None, names=["id", "source", "target", "scores", "label"])
    dev_df_lang = pd.read_csv(dev_file, sep="\t", header=None, names=["id", "source", "target", "scores", "label"])
    print(f"Loaded {train_file} with {len(train_df_lang)} examples")
    print(f"Loaded {dev_file} with {len(dev_df_lang)} examples")
    train_dfs.append(train_df_lang)
    dev_dfs.append(dev_df_lang)

train_df = pd.concat(train_dfs, ignore_index=True)
dev_df = pd.concat(dev_dfs, ignore_index=True)

train_df = shuffle(train_df, random_state=42)

# Check class distribution
train_percent = train_df["label"].value_counts() / len(train_df) * 100
dev_percent = dev_df["label"].value_counts() / len(dev_df) * 100

print("Training Class Percentage:\n", train_percent.round(2))
print("Dev Class Percentage:\n", dev_percent.round(2))


label_map = {"NOT": 0, "ERR": 1}
train_df["label"] = train_df["label"].map(label_map)
dev_df["label"] = dev_df["label"].map(label_map)

# Prepare input texts and labels
train_texts = (train_df["source"] + " [SEP] " + train_df["target"]).tolist()
train_labels = train_df["label"].tolist()
dev_texts = (dev_df["source"] + " [SEP] " + dev_df["target"]).tolist()
dev_labels = dev_df["label"].tolist()

print("************************************")
print(f"Number of training examples: {len(train_texts)}")
print(f"Number of development examples: {len(dev_texts)}")

Loaded enzh_majority_train.tsv with 6859 examples
Loaded enzh_majority_dev.tsv with 1000 examples
Training Class Percentage:
 label
NOT    83.82
ERR    16.18
Name: count, dtype: float64
Dev Class Percentage:
 label
NOT    85.9
ERR    14.1
Name: count, dtype: float64
************************************
Number of training examples: 6859
Number of development examples: 1000


### Transform data into tokens to be an input to the model


In [22]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
dev_encodings = tokenizer(dev_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

class ErrorDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ErrorDataset(train_encodings, train_labels)
dev_dataset = ErrorDataset(dev_encodings, dev_labels)

### Bert Model training 

In [23]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,  # Enable mixed precision training
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.409000,0.371739,0.857000,0.822215
2,0.349900,0.383206,0.842000,0.825697
3,0.235100,0.465417,0.822000,0.804493
4,0.187300,0.549238,0.823000,0.805166


TrainOutput(global_step=1716, training_loss=0.3139896012130595, metrics={'train_runtime': 778.4682, 'train_samples_per_second': 35.244, 'train_steps_per_second': 2.204, 'total_flos': 901495497636336.0, 'train_loss': 0.3139896012130595, 'epoch': 4.0})

### Evaluating against the dev_set

In [24]:
from sklearn.metrics import confusion_matrix
from datetime import datetime
from sklearn.metrics import matthews_corrcoef

eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

preds = trainer.predict(dev_dataset)
y_true = dev_labels
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["No Critical Error", "Critical Error"]))

mcc = matthews_corrcoef(y_true, y_pred)
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

# print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# out_file = f"predictions_{timestamp}.csv"
# pd.DataFrame({"text": dev_texts, "true_label": y_true, "pred_label": y_pred}).to_csv(out_file, index=False)

# print(f"Predictions saved to {out_file}")


Evaluation Results: {'eval_loss': 0.3832055628299713, 'eval_accuracy': 0.842, 'eval_f1': 0.8256969228941428, 'eval_runtime': 6.4724, 'eval_samples_per_second': 154.502, 'eval_steps_per_second': 4.944, 'epoch': 4.0}
                   precision    recall  f1-score   support

No Critical Error       0.88      0.94      0.91       859
   Critical Error       0.40      0.25      0.31       141

         accuracy                           0.84      1000
        macro avg       0.64      0.59      0.61      1000
     weighted avg       0.82      0.84      0.83      1000

Matthews Correlation Coefficient (MCC): 0.2318


BERT (distilbert-base-multilingual-cased)

Results:

Metric: "No Critical Error", "Critical Error"

Precision: 0.87, 0.56	
When the model predicts "Critical Error," it’s correct only 56% of the time.

Recall: 0.95, 0.32
The model misses 68% of actual critical errors (high false negatives).

F1-score: 0.91,	0.41
Poor balance for the minority class ("Critical Error").

Matthews Correlation Coefficient (MCC): 0.3441 -> a good metric for binary classification to show imbalance in data, indicating that there is a bias toward the majority class. Our model's predictions are a not that good (~35% better than random guessing).

We can see that the BERT model, is heavily affected by the imbalance in the dataset causing bias to the majority class (No critical error). The imbalance is caused by the Training data distribution: 

Class Percentage:
NOT    82.19
ERR    17.81

### Per language runs and results:
********************encs********************

Evaluation Results: {'eval_loss': 0.37832149863243103, 'eval_accuracy': 0.855, 'eval_f1': 0.8307146405507061, 'eval_runtime': 7.0829, 'eval_samples_per_second': 141.185, 'eval_steps_per_second': 4.518, 'epoch': 4.0}
                   precision    recall  f1-score   support

No Critical Error       0.87      0.97      0.92       840
   Critical Error       0.61      0.27      0.37       160

         accuracy                           0.85      1000
        macro avg       0.74      0.62      0.65      1000
     weighted avg       0.83      0.85      0.83      1000

Matthews Correlation Coefficient (MCC): 0.3360



********************ende********************

Evaluation Results: {'eval_loss': 0.5112722516059875, 'eval_accuracy': 0.768, 'eval_f1': 0.7619226611226613, 'eval_runtime': 6.3395, 'eval_samples_per_second': 157.741, 'eval_steps_per_second': 5.048, 'epoch': 4.0}
                   precision    recall  f1-score   support

No Critical Error       0.82      0.87      0.84       719
   Critical Error       0.60      0.51      0.55       281

         accuracy                           0.77      1000
        macro avg       0.71      0.69      0.70      1000
     weighted avg       0.76      0.77      0.76      1000

Matthews Correlation Coefficient (MCC): 0.4009


********************enja********************

Evaluation Results: {'eval_loss': 0.3551127314567566, 'eval_accuracy': 0.9, 'eval_f1': 0.8706100614204147, 'eval_runtime': 6.4958, 'eval_samples_per_second': 153.946, 'eval_steps_per_second': 4.926, 'epoch': 4.0}
                   precision    recall  f1-score   support

No Critical Error       0.91      0.99      0.95       904
   Critical Error       0.41      0.09      0.15        96

         accuracy                           0.90      1000
        macro avg       0.66      0.54      0.55      1000
     weighted avg       0.86      0.90      0.87      1000

Matthews Correlation Coefficient (MCC): 0.1594


********************enzh********************

Evaluation Results: {'eval_loss': 0.3832055628299713, 'eval_accuracy': 0.842, 'eval_f1': 0.8256969228941428, 'eval_runtime': 6.4724, 'eval_samples_per_second': 154.502, 'eval_steps_per_second': 4.944, 'epoch': 4.0}
                   precision    recall  f1-score   support

No Critical Error       0.88      0.94      0.91       859
   Critical Error       0.40      0.25      0.31       141

         accuracy                           0.84      1000
        macro avg       0.64      0.59      0.61      1000
     weighted avg       0.82      0.84      0.83      1000

Matthews Correlation Coefficient (MCC): 0.2318


### 3 Error Anaysis and Trustworthiness:

#### 3.1 Error types the model strugless with
**Bert**
* Translating non-european languages like japanese and chinses seems harder to the model.
* 

**General**:
These are the error types that the machine can't translate well and the models seems to struggle with:
* Non-toxic as toxic or vice verca (can't distinugish between actual toxicity/sarcsam and real emotions and sentiment)
* Cultural differences espcially on subtle matters (can't recognize cultural diversity and nuances)

How can we imporve this?
* Balance between the data the model is trained on for the Non critical and critical errors.
* Diverse data that represents a wide cultural differences and nuances for each language/culuture.
* Accuarate datset that differntiate between actual toxicity and emotions/ expresssions. (a lot of dataset has overrepresentation of toxicity)


#### 3.2
**BERT** 
pros:
* Can handle datasets that contains a lot of toxic data
* Size of data requires to fine-tune it is not large and scale well
* 
cons:
* Can't differentiate toxic from expressive emotions or sentiment
* Doesn't perform well with Multilanugal training (esp. with non-European languages)

Improvement?


Data Augmentation:
- Include more non-English examples and annotate for cultural context.
- Balance the dataset with nuanced cases (e.g., sarcasm, reclaimed slurs).

Evaluation Framework:
- Add metrics for fairness (e.g., disparity in FPR/FNR across demographics).
- Human-in-the-loop validation for ambiguous cases.

Bias Mitigation:
- Audit training data for underrepresented groups.
- Adopt adversarial debiasing techniques.
